2026.10.28 一致性修订：请 Restart Kernel & Run All。初始化会验证当前 embedding→特征→预测→概率；旧 niche 缓存不再复用，旧图输出已清空。Deployment 最佳模型用法保持不变。


# HCC Nature external validation — pan-cancer Hist2Pheno (UNI2 A2)

UNI2 A2 uses a 14 px crop (7 µm after rescale to 0.5 µm/px). Run `demo_uni2_A2.sh`. UNI predictions stay in `pred_pancancer_section`.


Seventeen H&E TIFFs under `data/Visium/HCC/nature_data` (plain `.tif` + `.ome.tif`), StarDist nuclei, **no cell-type ground truth** and no 10x JSON.
Predictions are the pan-cancer **cross-section** checkpoint
`A3_adjacent_spatial_k8mean_gate` (L1 / L12 / L2), spatial k=8.

This is **not** `External_val/HCC` (seven Visium JPEGs). µm/px is read **per slide**: OME `PhysicalSize` when it is a real scanner size (~0.501), else Aperio `MPP`, else 0.5. `L-8015` writes OME 84.67 µm/px (300 dpi) and that value is ignored.

```bash
cd /home/lingyu/ssd2/Python/Hist2Pheno
bash code/External_val/HCC_nature/demo_uni2_A2.sh dry-run
CUDA_DEVICE=0 bash code/External_val/HCC_nature/demo_uni2_A2.sh all
```

Outputs: `data/Visium/HCC/nature_data/pred_uni2_A2_bestmlp_paired_v2_seed42/`.
Screen output and elapsed time: `code/External_val/HCC_nature/HCC_nature.log`.

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42  # TrueType; PDF text stays editable
mpl.rcParams["ps.fonttype"] = 42


def save_figure(fig, path, *, dpi=None):
    """Write PDF (editable text) and a PNG preview. Call after the setup cell."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    dpi = int(SAVE_DPI if dpi is None else dpi)
    pdf = path.with_suffix(".pdf")
    png = path.with_suffix(".png")
    fig.savefig(pdf, format="pdf", bbox_inches="tight", facecolor="white", edgecolor="none")
    fig.savefig(png, format="png", dpi=dpi, bbox_inches="tight", facecolor="white", edgecolor="none")
    print(pdf)
    return pdf

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
import os
os.environ["UNI2_CKPT"] = str(REPO / "data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt")
STAR_PROB_MIN = 0.0  # 2026.10.28, set above 0 to store a separate _probXX folder
import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0" if float(STAR_PROB_MIN) <= 0 else f"{float(STAR_PROB_MIN):.2f}"
os.environ["UNI2_PRED_DIRNAME"] = "pred_uni2_A2_bestmlp_paired_v2_seed42"
sys.path.insert(0, str(REPO / "code"))

from Hist2Pheno_pkg.external_validation import simple_cohort

_cohort = simple_cohort("HCC_NATURE", repo=REPO, embedder="uni2")
PRED_ROOT = _cohort.pred_root
composition_table = _cohort.composition_table
discover_samples = _cohort.discover_samples
geometry = _cohort.geometry
sample_paths = _cohort.sample_paths
ensure_l2_probs = _cohort.ensure_l2_probs
plot_spatial = _cohort.plot_spatial
plot_composition = _cohort.plot_composition
# Re-run the plot cell after changing these. Notebook display uses DPI; PNG uses SAVE_DPI; PDF is Type 42 vector.
SAVE_FIGURES = True
FIGSIZE = (17.2, 6.6)
DPI = 200
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 1.0

samples = discover_samples()
for sample in samples:
    g = geometry(sample)
    print(
        f"{sample}  mpp={g['mpp_um']:.4f} µm/px ({g.get('mpp_source', '')})  "
        f"native14={g['patch_um_if_native_16px']:.2f} µm  "
        f"after rescale={g['patch_um_after_rescale']:.1f} µm  "
        f"scale={g['scale']:.4f}"
    )
    p = sample_paths(sample)
    print(f"  pred exists: {p['pred'].is_file()}  {p['pred']}")
# 2026.10.28, mandatory current-input preflight / 后续直接读 h5ad 的图表也不得绕过上游
import os as _pipeline_os
_pipeline_os.environ["STAR_PROB_MIN"] = str(globals().get("STAR_PROB_MIN", 0.0))
if "ensure_l2_probs" in globals():
    _owner = ensure_l2_probs.__self__
    _current_samples = list(samples) if "samples" in globals() else _owner.discover_samples()
    _owner.cmd_predict(_current_samples, force=False, allow_cpu=True)
elif "ue" in globals():
    ue.cmd_predict(COHORT, samples, cuda_device=_pipeline_os.environ.get("CUDA_DEVICE", "0"),
                   force=False, allow_cpu=True, embedder=EMBEDDER)
else:
    raise RuntimeError("No current prediction validator; run the setup cell first")


In [ ]:
import pandas as pd
from IPython.display import display

missing = [s for s in samples if not sample_paths(s)["pred"].is_file()]
if missing:
    raise FileNotFoundError(
        "No predictions yet for "
        + ", ".join(missing)
        + ". Run: bash code/External_val/HCC_nature/demo_uni2_A2.sh all"
    )

comp = composition_table(samples)

for head in ("L1", "L12", "L2"):
    sub = comp.loc[comp["head"] == head].copy()
    order = sub.groupby("class")["fraction"].max().sort_values(ascending=False).index
    wide = (
        sub.pivot_table(index="class", columns="sample", values=["fraction", "n", "mean_conf"])
        .reindex(order)
    )
    print(f"\n===== {head} =====")
    display(wide.round(3))

In [ ]:
import matplotlib.pyplot as plt


SAVE_FIGURES = True
SAVE_PDF = True  # Type 42 PDF; set False for PNG only
FIGSIZE = (17.2, 6.6)
DPI = 300
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 2.0

for fig in plot_spatial(
    ["L-8012_HE.ome"],
    figsize=FIGSIZE,
    dpi=DPI,
    he_max_side=HE_MAX_SIDE,
    marker_px=MARKER_PX,
    save=SAVE_FIGURES,
    save_pdf=SAVE_PDF,
    save_dpi=SAVE_DPI,
):
    display(fig)
    plt.close(fig)

## Nature 2025 TIMES paper — what Hist2Pheno can reuse

Source: Jia *et al.*, *Nature* (2025) [Spatial immune scoring system predicts hepatocellular carcinoma recurrence](https://www.nature.com/articles/s41586-025-08668-x).
The URL `/figures/6` is **Extended Data Fig. 1** (the article has main Figs. 1–5). Panel **a** is the H&E / GPC3 / CD34 / GS / Reticulin panel that marks **adjacent stroma (AS)**, **invasive front (IF)** and **tumour centre (TC)**; high-resolution extras are on [OSF](https://osf.io/db3uf/). The schematic you attached is L-8012 GPC3 + Reticulin zooms from that panel (IF above the dashed curve, TC below, 400 µm bar).

**What the paper did.** After resection, HCC recurs often. In 61 patients, NK cells at IF vs TC track recurrence. They built **TIMES** from the *spatial* pattern of five markers (SPON2, ZFP36L2, ZFP36, VIM, HLA-DRB1); the score beat TNM/BCLC (HR 88.2) and held in 231 patients from five cohorts (82.2% accuracy). SPON2 on NK cells increases IFNγ and NK influx at the IF. Modalities: Visium, DSP, mIHC, LC–MS/MS, mouse models.

**What this 17-slide H&E set can add (no IHC / Visium GT here).**

| Paper readout | Hist2Pheno analogue |
|--|--|
| Pathologist IF / TC from GPC3 + Reticulin | Geometric AS / IF / TC: stromal cap = AS, ~160 µm band on the dashed tumour–stroma contour = IF, core = TC |
| NK density at IF vs TC (recurrence) | L2 `NK` counts in the same compartments |
| TIMES 5-gene spatial score | **Cannot** be read from H&E. Use cell-type neighbourhoods instead |
| Immune context at IF | L2 CD4/CD8/Treg, B/Plasma, DC, macrophage — broader than the paper’s NK focus |
| — | TLS / SRI / TNI on the IF–TC band (already in this notebook) |

Practical tests on these slides: (1) does the predicted tumour/stroma boundary recover the published IF–TC curve; (2) is predicted NK (and other immune) enriched at IF vs TC; (3) if REC / non-REC labels exist for L-8001…L-8017, does H&E-only IF immune composition separate them. Next cell crops that L-8012 zoom and overlays Hist2Pheno.


In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Rectangle

from Hist2Pheno_pkg.panel_figures import crop_he_rectangle, plot_roi_pred_spatial

SAMPLE_FIG6A = "L-8012_HE.ome"
# Native HE pixels. Same left edge as the first crop; right edge extended to include more TC.
# Matches Extended Data Fig. 1a GPC3/Reticulin zoom (400 µm bar in the paper).
ROI_X0, ROI_Y0, ROI_X1, ROI_Y1 = 1301, 5463, 9167, 10666
SCALE_BAR_UM = 400.0

paths = sample_paths(SAMPLE_FIG6A)
pred = ad.read_h5ad(paths["pred"])
xy = np.asarray(pred.obsm["spatial_HE"], dtype=np.float64)
mpp = float(geometry(SAMPLE_FIG6A)["mpp_um"])
in_roi = (
    (xy[:, 0] >= ROI_X0)
    & (xy[:, 0] < ROI_X1)
    & (xy[:, 1] >= ROI_Y0)
    & (xy[:, 1] < ROI_Y1)
)
w_um = (ROI_X1 - ROI_X0) * mpp
h_um = (ROI_Y1 - ROI_Y0) * mpp
print(
    f"{SAMPLE_FIG6A}  ROI [{ROI_X0}:{ROI_X1}]×[{ROI_Y0}:{ROI_Y1}]  "
    f"{w_um:.0f}×{h_um:.0f} µm  nuclei={int(in_roi.sum()):,} / {len(xy):,}  "
    f"µm/px={mpp:.4f}"
)

thumb, full_w, full_h = _cohort._he_thumbnail(paths["he"], max_side=HE_MAX_SIDE)
he_roi = crop_he_rectangle(paths["he"], ROI_X0, ROI_Y0, ROI_X1, ROI_Y1)
roi_h, roi_w = he_roi.shape[:2]
bar_px = SCALE_BAR_UM / mpp

fig_he, axes = plt.subplots(1, 2, figsize=(12.4, 6.2), dpi=DPI)
axes[0].imshow(thumb, extent=(0, full_w, full_h, 0), interpolation="bilinear")
axes[0].add_patch(
    Rectangle(
        (ROI_X0, ROI_Y0),
        ROI_X1 - ROI_X0,
        ROI_Y1 - ROI_Y0,
        fill=False,
        edgecolor="black",
        linewidth=1.6,
    )
)
axes[0].set_title(f"{SAMPLE_FIG6A}  overview + ED Fig. 1a zoom box")
axes[1].imshow(he_roi)
pad = 40
x1 = roi_w - pad
y1 = roi_h - pad
axes[1].plot([x1 - bar_px, x1], [y1, y1], color="black", lw=2.4, solid_capstyle="butt")
axes[1].text(
    x1 - 0.5 * bar_px,
    y1 - 18,
    f"{SCALE_BAR_UM:.0f} µm",
    ha="center",
    va="bottom",
    fontsize=10,
    color="black",
)
axes[1].set_title("zoom-in HE  (IF top / TC bottom)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
fig_he.tight_layout()
fig_dir = PRED_ROOT / "figures"
if SAVE_FIGURES:
    fig_dir.mkdir(parents=True, exist_ok=True)
    save_figure(fig_he, fig_dir / f"{SAMPLE_FIG6A}_EDFig1a_HE_zoom.pdf")
display(fig_he)
plt.close(fig_he)


In [ ]:

fig_pred, _ = plot_roi_pred_spatial(
    pred,
    paths["he"],
    x0=ROI_X0,
    y0=ROI_Y0,
    x1=ROI_X1,
    y1=ROI_Y1,
    palettes=_cohort._palette(),
    in_roi=in_roi,
    marker_px=max(float(MARKER_PX), 5.0),
    dpi=DPI,
    figsize=(17.2, 12.5),
    title=(
        f"{SAMPLE_FIG6A}  ED Fig. 1a IF–TC zoom  "
        f"Hist2Pheno L1/L12/L2  n={int(in_roi.sum()):,}"
    ),
    save_path=None,
    save_dpi=SAVE_DPI,
)
if SAVE_FIGURES:
    save_figure(fig_pred, fig_dir / f"{SAMPLE_FIG6A}_EDFig1a_pred_zoom.pdf")
display(fig_pred)
plt.close(fig_pred)


In [ ]:
# import matplotlib.pyplot as plt

# for fig in plot_spatial(
#     samples,
#     figsize=FIGSIZE,
#     dpi=DPI,
#     he_max_side=HE_MAX_SIDE,
#     marker_px=MARKER_PX,
#     save=SAVE_FIGURES,
#     save_dpi=SAVE_DPI,
# ):
#     display(fig)
#     plt.close(fig)

In [ ]:
# for fig in plot_composition(samples, save=SAVE_FIGURES, dpi=DPI, save_dpi=SAVE_DPI):
#     display(fig)
#     plt.close(fig)

## Histology-derived niche indices (TLS / SRI / TNI)

Same implementation as `code/Xenium_lung/histology_derived_niche_index.ipynb` §3, via `Hist2Pheno_pkg/histology_niche_index.py`. Lung ARI / FRI are not used.

Scores use **L2 softmax**, not the argmax label. `ensure_l2_probs` caches `{sample}_l2_probs.npz`.

| Index | Abundance | Spatial (radius R) |
|--|--|--|
| **TLS** | sum(B_cell + Plasma + CD4_T + CD8_T + Tregs + DC) | `(B_local × T_local × DC_local)^(1/3)` |
| **SRI** | (Fibroblast + Myofibroblast + Stromal) / epithelium | Fib_local / Epi_local |
| **TNI** | (fibroblast + Macrophage + Endothelial) / epithelium | `(Fib × Mac × Endo)^(1/3) / Epi_local` |

Epithelium = Epithelial, Tumor, Alveolar, Injury_epithelial, DCIS, Myoepithelial. NK is left out of TLS. `spatial_HE` is native TIFF pixels, so R = **55 µm** is converted with **that slide's** µm/px (OME / Aperio / fallback), not with a global 0.5 unless that is the value for the slide. TLS, SRI, and TNI each use a 1×3 row: abundance, spatial score without markers, then the same spatial map with cyan candidates.

In [ ]:
# import importlib

# for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "Xenium_lung"):
#     if str(_p) not in sys.path:
#         sys.path.insert(0, str(_p))

# import histology_niche_index as hni
# importlib.reload(hni)
# import histology_derived_niche_index as hdni
# importlib.reload(hdni)
# from histology_niche_index import OrganNicheConfig

# import anndata as ad

# TLS_RADIUS_UM = 55.0
# RECOMPUTE_NICHE = False

# HCC_NATURE_NICHE = OrganNicheConfig(
#     pan_organ="pancancer_hcc_nature",
#     tls_b=("B_cell", "Plasma"),
#     tls_t=("CD4_T", "CD8_T", "Tregs"),
#     tls_dc=("DC",),
#     epithelium=(
#         "Epithelial", "Tumor", "Alveolar",
#         "Injury_epithelial", "DCIS", "Myoepithelial",
#     ),
#     fibroblast=("Fibroblast", "Myofibroblast", "Stromal"),
#     endothelial=("Endothelial",),
#     macrophage=("Macrophage",),
#     um_per_he_pixel=float(geometry(samples[0])["mpp_um"]),
#     spatial_radius_um=TLS_RADIUS_UM,
#     cohort="HCC_nature",
# )
# display(hni.index_formula_table(HCC_NATURE_NICHE))

In [ ]:
# import numpy as np
# niche_frames = {}
# for sample in samples:
#     paths = sample_paths(sample)
#     niche_path = paths["pred"].with_name(f"{sample}_niche_indices.h5ad")
#     mpp = float(geometry(sample)["mpp_um"])
#     if niche_path.is_file() and not RECOMPUTE_NICHE:
#         cached = ad.read_h5ad(niche_path)
#         df = cached.obs.copy()
#         xy = np.asarray(cached.obsm["spatial_HE"], dtype=np.float64)
#         df["coord_x"] = xy[:, 0]
#         df["coord_y"] = xy[:, 1]
#         print(f"[niche] cache {niche_path.name}  n={len(df):,}  µm/px={mpp:.4f}")
#     else:
#         probs, class_names = ensure_l2_probs(sample)
#         missing = hni.missing_index_cell_types(class_names, HCC_NATURE_NICHE)
#         if missing:
#             raise RuntimeError(f"{sample} missing niche compartments: {missing}")
#         pred = ad.read_h5ad(paths["pred"])
#         coords = np.asarray(pred.obsm["spatial_HE"], dtype=np.float64)
#         df = pred.obs.copy()
#         df["coord_x"] = coords[:, 0]
#         df["coord_y"] = coords[:, 1]
#         df = hni.add_abundance_indices(df, probs, class_names, HCC_NATURE_NICHE)
#         df = hni.add_spatial_indices(
#             df, probs, class_names, coords, HCC_NATURE_NICHE,
#             radius_um=TLS_RADIUS_UM,
#             um_per_pixel=mpp,
#         )
#         out = ad.AnnData(obs=df.drop(columns=["coord_x", "coord_y"]))
#         out.obsm["spatial_HE"] = coords
#         out.uns["tls_radius_um"] = TLS_RADIUS_UM
#         out.uns["um_per_px"] = mpp
#         __import__("pipeline_provenance").atomic_h5ad(out,niche_path)
#         print(f"[niche] wrote {niche_path}  n={len(df):,}")
#     niche_frames[sample] = df
#     print(f"  TLS hotspot: {hni.summarize_tls_hotspots(df)}")
#     print(
#         f"  mean TLS={df['idx_TLS'].mean():.3f}  TLS_spatial={df['idx_TLS_spatial'].mean():.3f}  "
#         f"SRI={df['idx_SRI_ratio'].mean():.3f}  TNI={df['idx_TNI_ratio'].mean():.3f}"
#     )

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd
# from IPython.display import display

# fig_dir = PRED_ROOT / "figures"
# fig_dir.mkdir(parents=True, exist_ok=True)


# def tls_score_row(df, sample: str) -> dict:
#     a = df["idx_TLS"].to_numpy(dtype=float)
#     s = df["idx_TLS_spatial"].to_numpy(dtype=float)
#     a_hi = a >= np.percentile(a, 95)
#     s_hi = df["tls_candidate"].fillna(False).astype(bool).to_numpy()
#     inter = int(np.logical_and(a_hi, s_hi).sum())
#     union = int(np.logical_or(a_hi, s_hi).sum())
#     n_a = int(a_hi.sum())
#     return {
#         "sample": sample,
#         "n": int(len(df)),
#         "TLS_mean": float(np.mean(a)),
#         "TLS_median": float(np.median(a)),
#         "TLS_p95": float(np.percentile(a, 95)),
#         "TLS_max": float(np.max(a)),
#         "TLS_spatial_mean": float(np.mean(s)),
#         "TLS_spatial_median": float(np.median(s)),
#         "TLS_spatial_p95": float(np.percentile(s, 95)),
#         "TLS_spatial_max": float(np.max(s)),
#         "pearson": float(np.corrcoef(a, s)[0, 1]),
#         "spearman": float(pd.Series(a).corr(pd.Series(s), method="spearman")),
#         "top5_jaccard": (inter / union) if union else float("nan"),
#         "abundance_top5_also_spatial_top5": (inter / n_a) if n_a else float("nan"),
#         "TLS_spatial_mean_in_abundance_top5": float(s[a_hi].mean()) if n_a else float("nan"),
#         "TLS_mean_in_spatial_top5": float(a[s_hi].mean()) if s_hi.any() else float("nan"),
#     }


# quant_rows = [tls_score_row(df, sample) for sample, df in niche_frames.items()]
# tls_quant = pd.DataFrame(quant_rows)
# tls_quant_path = fig_dir / "tls_score_summary.csv"
# tls_quant.to_csv(tls_quant_path, index=False)
# print(tls_quant_path)
# display(tls_quant.round(3))

# for sample, df in niche_frames.items():
#     fig, axes = hdni.plot_tls_abundance_vs_spatial(
#         df,
#         sample,
#         radius_um=TLS_RADIUS_UM,
#         use_entropy=False,
#         split_spatial_candidates=True,
#         dpi=DPI,
#         save_dpi=SAVE_DPI,
#         point_size=0.15,
#         candidate_size=1.2,
#         save_path=(fig_dir / f"{sample}_TLS_abundance_vs_spatial.png") if SAVE_FIGURES else None,
#     )
#     display(fig)
#     plt.close(fig)

#     for abundance_col, spatial_col, candidate_col, tag in (
#         ("idx_SRI_ratio", "idx_SRI_spatial", "sri_candidate", "SRI"),
#         ("idx_TNI_ratio", "idx_TNI_spatial", "tni_candidate", "TNI"),
#     ):
#         fig, axes = hdni.plot_tls_abundance_vs_spatial(
#             df,
#             sample,
#             radius_um=TLS_RADIUS_UM,
#             abundance_col=abundance_col,
#             spatial_col=spatial_col,
#             candidate_col=candidate_col,
#             score_name=tag,
#             use_entropy=False,
#             split_spatial_candidates=True,
#             dpi=DPI,
#             save_dpi=SAVE_DPI,
#             point_size=0.15,
#             candidate_size=1.2,
#             save_path=(fig_dir / f"{sample}_{tag}_abundance_vs_spatial.png") if SAVE_FIGURES else None,
#         )
#         display(fig)
#         plt.close(fig)

## Hist2Pheno implementation of the TIMES-paper readouts

The paper (Jia *et al.*, *Nature* 2025) uses IHC / Visium / DSP. This section does only what **H&E + Hist2Pheno L2** can do. TIMES genes and REC / non-REC labels are **not** in `nature_data`.

**Rule.** Bin nuclei (`BIN_UM` ≈ 80 µm), smooth the L2 tumour/epithelium fraction, fill a tumour domain, then draw the **dashed IF–TC contour** as that domain’s edge (same role as the GPC3/Reticulin dashed line).

| Compartment | Definition |
|--|--|
| **IF** | A thin band on the dashed line: `BAND_UM` (160 µm) inside the tumour plus `AS_SIDE_UM` (80 µm) into the cap |
| **TC** | Tumour domain inside the inner edge of that band |
| **AS** | Tissue outside the outer edge (stromal cap / septum) |

A 300 µm *inside-only* rim painted most of the L-8012 nodules orange. The thin two-sided band is meant to sit on the published curve. Tune `BAND_UM` / `AS_SIDE_UM` / `TUMOR_BIN_MIN` in the next cell.

**Immune.** L2 `NK` is almost absent on L-8012, so the paper’s NK-at-IF test cannot run. The immune panel uses classes that are actually predicted: CD4 / CD8 / Treg / B / Plasma / DC, plus a TLS-like sum. Compare **AS vs IF vs TC** (lymphocytes live in the cap, not only IF vs TC).


In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.ndimage import (
    binary_closing,
    binary_dilation,
    binary_erosion,
    binary_fill_holes,
    binary_opening,
    gaussian_filter,
)

from Hist2Pheno_pkg.panel_figures import crop_he_rectangle

SAMPLE_FIG6A = "L-8012_HE.ome"
ROI_X0, ROI_Y0, ROI_X1, ROI_Y1 = 1301, 5463, 9167, 10666
BIN_UM = 80.0
BAND_UM = 160.0
AS_SIDE_UM = 80.0
TUMOR_BIN_MIN = 0.10
COMP_ORDER = ("AS", "IF", "TC")
COMP_COLOR = {"AS": "#bdbdbd", "IF": "#e07a3d", "TC": "#c44e52"}
TUMOR_L2 = (
    "Tumor", "Epithelial", "Alveolar",
    "Injury_epithelial", "DCIS", "Myoepithelial",
)
IMMUNE_L2 = ("NK", "CD4_T", "CD8_T", "Tregs", "B_cell", "Plasma", "DC")
FOCUS_L2 = IMMUNE_L2 + ("Macrophage", "Tumor", "Epithelial", "Fibroblast", "Stromal")
TLS_L2 = ("B_cell", "Plasma", "CD4_T", "CD8_T", "Tregs", "DC")
IMMUNE_COLOR = {
    "NK": "#6a3d9a",
    "CD4_T": "#1f78b4",
    "CD8_T": "#33a02c",
    "Tregs": "#a6cee3",
    "B_cell": "#ff7f00",
    "Plasma": "#b15928",
    "DC": "#e31a1c",
}


def assign_if_band(
    xy: np.ndarray,
    is_tumor: np.ndarray,
    mpp: float,
    *,
    origin_xy=None,
    size_xy=None,
    bin_um: float = BIN_UM,
    band_um: float = BAND_UM,
    as_side_um: float = AS_SIDE_UM,
    tumor_bin_min: float = TUMOR_BIN_MIN,
) -> tuple[np.ndarray, dict]:
    """AS / IF / TC: thin band on the tumour–stroma contour (paper dashed line)."""
    xy = np.asarray(xy, dtype=np.float64)
    is_tumor = np.asarray(is_tumor, dtype=bool)
    if origin_xy is None:
        origin_xy = xy.min(axis=0)
    if size_xy is None:
        size_xy = xy.max(axis=0) - origin_xy
    ox, oy = map(float, origin_xy)
    sx, sy = map(float, size_xy)
    bin_px = float(bin_um) / float(mpp)
    nx = max(1, int(np.ceil(sx / bin_px)))
    ny = max(1, int(np.ceil(sy / bin_px)))
    ix = np.clip(((xy[:, 0] - ox) / bin_px).astype(int), 0, nx - 1)
    iy = np.clip(((xy[:, 1] - oy) / bin_px).astype(int), 0, ny - 1)
    n_bin = np.zeros((ny, nx), dtype=np.float64)
    t_bin = np.zeros((ny, nx), dtype=np.float64)
    np.add.at(n_bin, (iy, ix), 1.0)
    np.add.at(t_bin, (iy, ix), is_tumor.astype(np.float64))
    f = np.divide(t_bin, n_bin, out=np.zeros_like(t_bin), where=n_bin >= 6)
    fs = gaussian_filter(f, sigma=1.5)
    populated = n_bin >= 6
    tumor_dom = binary_fill_holes((fs >= tumor_bin_min) & populated)
    if tumor_dom.sum() < 12:
        thr = float(np.percentile(fs[populated], 60)) if populated.any() else tumor_bin_min
        tumor_dom = binary_fill_holes((fs >= thr) & populated)
    tumor_dom = binary_fill_holes(binary_opening(binary_closing(tumor_dom, iterations=2), iterations=1))
    steps_in = max(1, int(round(float(band_um) / float(bin_um))))
    steps_out = max(1, int(round(float(as_side_um) / float(bin_um))))
    tc_dom = binary_erosion(tumor_dom, iterations=steps_in)
    if int(tc_dom.sum()) < 8 and steps_in > 1:
        steps_in = steps_in - 1
        tc_dom = binary_erosion(tumor_dom, iterations=steps_in)
    outer = binary_dilation(tumor_dom, iterations=steps_out)
    if_dom = outer & ~tc_dom
    as_dom = populated & ~outer
    label = np.zeros((ny, nx), dtype=np.int8)
    label[as_dom] = 1
    label[if_dom] = 2
    label[tc_dom] = 3
    names = np.array(["AS", "AS", "IF", "TC"], dtype=object)
    meta = {
        "tumor_dom": tumor_dom,
        "bin_px": bin_px,
        "origin_xy": (ox, oy),
        "steps_in": steps_in,
        "steps_out": steps_out,
    }
    return names[label[iy, ix]], meta


def compartment_counts(labels: np.ndarray, compartments: np.ndarray) -> pd.DataFrame:
    rows = []
    for name in COMP_ORDER:
        hit = compartments == name
        n = int(hit.sum())
        if n == 0:
            continue
        vc = pd.Series(labels[hit]).value_counts()
        for cls, k in vc.items():
            rows.append(
                {
                    "compartment": name,
                    "class": str(cls),
                    "n": int(k),
                    "fraction": float(k) / float(n),
                    "n_comp": n,
                }
            )
    return pd.DataFrame(rows)


def _bars(ax, counts: pd.DataFrame, classes: list[str], title: str) -> None:
    x = np.arange(len(classes))
    width = 0.26
    for i, name in enumerate(COMP_ORDER):
        sub = counts.loc[counts["compartment"] == name].set_index("class")
        vals = [float(sub.loc[c, "fraction"]) if c in sub.index else 0.0 for c in classes]
        ax.bar(x + (i - 1) * width, vals, width=width, color=COMP_COLOR[name], label=name)
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=55, ha="right", fontsize=8)
    ax.set_ylabel("fraction of nuclei in compartment")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)


fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

paths = sample_paths(SAMPLE_FIG6A)
pred = ad.read_h5ad(paths["pred"])
xy = np.asarray(pred.obsm["spatial_HE"], dtype=np.float64)
l2 = pred.obs["pred_CT"].astype(str).to_numpy()
mpp = float(geometry(SAMPLE_FIG6A)["mpp_um"])
n_nk_slide = int((l2 == "NK").sum())
print(f"{SAMPLE_FIG6A}  whole slide L2 NK={n_nk_slide:,}  / {len(l2):,}")

in_roi = (
    (xy[:, 0] >= ROI_X0)
    & (xy[:, 0] < ROI_X1)
    & (xy[:, 1] >= ROI_Y0)
    & (xy[:, 1] < ROI_Y1)
)
xy_r = xy[in_roi]
l2_r = l2[in_roi]
comp, meta = assign_if_band(
    xy_r,
    np.isin(l2_r, TUMOR_L2),
    mpp,
    origin_xy=(ROI_X0, ROI_Y0),
    size_xy=(ROI_X1 - ROI_X0, ROI_Y1 - ROI_Y0),
)
print(
    f"zoom  BAND={BAND_UM:.0f} µm in + {AS_SIDE_UM:.0f} µm AS-side  "
    + "  ".join(f"{c}={(comp == c).sum():,}" for c in COMP_ORDER)
)
if n_nk_slide < 20:
    print(
        f"NOTE: L2 NK is essentially absent on this slide (n={n_nk_slide}). "
        "Read CD4/CD8/B on AS vs IF vs TC, not NK."
    )

counts = compartment_counts(l2_r, comp)
wide = (
    counts.pivot_table(index="class", columns="compartment", values="fraction")
    .reindex(columns=list(COMP_ORDER))
)
focus = [c for c in FOCUS_L2 if c in set(wide.index)]
focus += [c for c in wide.index if c not in focus]
wide = wide.reindex(focus)
tls = counts.loc[counts["class"].isin(TLS_L2)].groupby("compartment")["fraction"].sum()
print("  TLS-like  " + "  ".join(f"{c}={tls.get(c, 0):.3f}" for c in COMP_ORDER))
display(wide.round(3))

he_roi = crop_he_rectangle(paths["he"], ROI_X0, ROI_Y0, ROI_X1, ROI_Y1)
roi_h, roi_w = he_roi.shape[:2]
xy_loc = np.column_stack([xy_r[:, 0] - ROI_X0, xy_r[:, 1] - ROI_Y0])
point_s = (max(float(MARKER_PX), 2.0) * 72.0 / float(DPI)) ** 2
imm_s = (max(float(MARKER_PX), 5.0) * 72.0 / float(DPI)) ** 2
ny, nx = meta["tumor_dom"].shape
xs = (np.arange(nx) + 0.5) * meta["bin_px"]
ys = (np.arange(ny) + 0.5) * meta["bin_px"]

fig, axes = plt.subplots(2, 2, figsize=(10.8, 10.2), dpi=DPI)
axes[0, 0].imshow(he_roi, extent=(0, roi_w, roi_h, 0), interpolation="bilinear")
for name in COMP_ORDER:
    hit = comp == name
    axes[0, 0].scatter(
        xy_loc[hit, 0],
        xy_loc[hit, 1],
        c=[COMP_COLOR[name]],
        s=5*point_s,
        linewidths=0,
        rasterized=True,
        label=f"{name}  n={int(hit.sum()):,}",
    )
axes[0, 0].contour(
    xs,
    ys,
    meta["tumor_dom"].astype(float),
    levels=[0.5],
    colors="black",
    linestyles="--",
    linewidths=1.35,
)
axes[0, 0].set_xlim(0, roi_w)
axes[0, 0].set_ylim(roi_h, 0)
axes[0, 0].set_xticks([])
axes[0, 0].set_yticks([])
axes[0, 0].set_aspect("equal")
axes[0, 0].set_title("AS / IF / TC + dashed IF–TC contour")
axes[0, 0].legend(loc="lower right", fontsize=16, markerscale=10, frameon=True, framealpha=0.92, handletextpad=0.45, borderpad=0.7, labelspacing=0.55)

axes[0, 1].imshow(he_roi, extent=(0, roi_w, roi_h, 0), interpolation="bilinear")
axes[0, 1].contour(
    xs,
    ys,
    meta["tumor_dom"].astype(float),
    levels=[0.5],
    colors="black",
    linestyles="--",
    linewidths=1.35,
)
for cls in IMMUNE_L2:
    hit = l2_r == cls
    if not hit.any():
        continue
    axes[0, 1].scatter(
        xy_loc[hit, 0],
        xy_loc[hit, 1],
        c=[IMMUNE_COLOR[cls]],
        s=5*imm_s,
        linewidths=0,
        rasterized=True,
        label=f"{cls}  n={int(hit.sum()):,}",
    )
axes[0, 1].set_xlim(0, roi_w)
axes[0, 1].set_ylim(roi_h, 0)
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])
axes[0, 1].set_aspect("equal")
axes[0, 1].set_title("L2 immune on HE  (NK nearly absent)")
axes[0, 1].legend(loc="lower right", fontsize=16, markerscale=5, frameon=True, framealpha=0.92, handletextpad=0.45, borderpad=0.7, labelspacing=0.45)

plot_all = [c for c in FOCUS_L2 if c in set(counts["class"])]
_bars(axes[1, 0], counts, plot_all, "L2  AS vs IF vs TC")
plot_imm = [c for c in IMMUNE_L2 if c in set(counts["class"])]
_bars(axes[1, 1], counts, plot_imm, "Immune only  AS vs IF vs TC")
fig.suptitle(
    f"{SAMPLE_FIG6A}  dashed contour + immune  (BAND={BAND_UM:.0f} µm, not IHC)",
    fontsize=12,
)
fig.tight_layout()
if SAVE_FIGURES:
    save_figure(fig, fig_dir / f"{SAMPLE_FIG6A}_IF_TC_hist2pheno.pdf")
display(fig)
plt.close(fig)


### How to read the L-8012 figure (`L-8012_HE.ome_IF_TC_hist2pheno.png`)

Four panels, same Extended Data Fig. 1a zoom. Saved under `pred_uni2_A2_bestmlp_paired_v2_seed42/figures/`.

**Top left — compartments + dashed contour.** Grey **AS** is the stromal cap and the fibrous septum (n ≈ 23.8k). Orange **IF** is the thin band on the black dashed line (n ≈ 17.0k). Red **TC** is the interior of both nodules (n ≈ 19.8k). The dashed line is the edge of the *smoothed L2 tumour-fraction domain*, not a pathologist GPC3/Reticulin trace. It wraps both lobes as one polygon, so it is coarser than the paper’s tight capsule curve, but the orange rim now sits on that border instead of flooding the whole mass.

**Top right — predicted immune on HE.** Counts in this zoom: NK 2, CD4_T 453, CD8_T 74, Tregs 19, B_cell 447, Plasma 87, DC 24. Almost all of these dots lie **above the dashed line** (AS / cap). The nodule interior is nearly empty of lymphocytes. NK is not usable here (2 nuclei).

**Bottom left — all L2, AS vs IF vs TC.** The nodule is called Macrophage + Fibroblast + Tumor, not a pure epithelial core. Tumour fraction rises toward the core (TC > IF ≫ AS). Macrophage is highest in IF. Fibroblast is high in all three. Stromal is mainly AS. This is a Hist2Pheno phenotype of this H&E, not IHC.

**Bottom right — immune only (note the y-axis, max ≈ 1.8%).** CD4 and B are almost entirely in **AS** (~1.8% and ~1.7% of AS nuclei). IF is an order of magnitude lower; TC is near zero. NK is blank. So on this slide the H&E-only immune contrast is **cap (AS) ≫ rim (IF) > core (TC)** for T and B, not the paper’s “NK high at IF”.

**What this figure supports.** (1) A geometric AS / IF / TC sketch from H&E that follows the published IF–TC idea. (2) Predicted T/B live in the cap, not the core.  
**What it does not support.** TIMES, SPON2+ NK, or recurrence. Do not treat the dashed polygon as the official pathologist outline.


In [ ]:
# All 17 slides: same thin band on the dashed contour. No REC / non-REC labels.

rows = []
for sample in samples:
    p = sample_paths(sample)
    pred_i = ad.read_h5ad(p["pred"])
    xy_i = np.asarray(pred_i.obsm["spatial_HE"], dtype=np.float64)
    l2_i = pred_i.obs["pred_CT"].astype(str).to_numpy()
    mpp_i = float(geometry(sample)["mpp_um"])
    comp_i, _ = assign_if_band(xy_i, np.isin(l2_i, TUMOR_L2), mpp_i)
    n_if = int((comp_i == "IF").sum())
    n_tc = int((comp_i == "TC").sum())
    n_as = int((comp_i == "AS").sum())
    n_nk = int((l2_i == "NK").sum())
    row = {
        "sample": sample,
        "n": int(len(l2_i)),
        "n_NK": n_nk,
        "n_AS": n_as,
        "n_IF": n_if,
        "n_TC": n_tc,
        "frac_AS": n_as / len(l2_i),
        "frac_IF": n_if / len(l2_i),
        "frac_TC": n_tc / len(l2_i),
    }
    for tag, names in (
        ("NK", ("NK",)),
        ("T", ("CD4_T", "CD8_T", "Tregs")),
        ("B_plasma", ("B_cell", "Plasma")),
        ("myeloid", ("Macrophage", "DC", "Neutrophil")),
        ("TLS_like", TLS_L2),
        ("tumor", TUMOR_L2),
        ("stroma", ("Fibroblast", "Myofibroblast", "Stromal")),
    ):
        is_cls = np.isin(l2_i, names)
        for comp_name in COMP_ORDER:
            n_comp = int((comp_i == comp_name).sum())
            row[f"{tag}_{comp_name}"] = (
                float(is_cls[comp_i == comp_name].mean()) if n_comp else np.nan
            )
    rows.append(row)
    print(
        f"{sample:18s}  AS={n_as:7,}  IF={n_if:7,}  TC={n_tc:7,}  "
        f"NK={n_nk:5,}  T_AS={row['T_AS']:.4f}  T_IF={row['T_IF']:.4f}  "
        f"B_AS={row['B_plasma_AS']:.4f}  B_IF={row['B_plasma_IF']:.4f}"
    )

if_tc = pd.DataFrame(rows)
if_tc_path = fig_dir / "HCC_nature_IF_TC_L2_summary.csv"
if_tc.to_csv(if_tc_path, index=False)
print(if_tc_path)
print(
    f"slides with L2 NK < 20: {int((if_tc['n_NK'] < 20).sum())} / {len(if_tc)}  "
    "— do not interpret NK IF/TC on those slides"
)
display(
    if_tc[
        [
            "sample", "n_NK", "n_AS", "n_IF", "n_TC",
            "T_AS", "T_IF", "T_TC",
            "B_plasma_AS", "B_plasma_IF", "B_plasma_TC",
            "TLS_like_AS", "TLS_like_IF", "TLS_like_TC",
            "NK_AS", "NK_IF", "NK_TC",
        ]
    ].round(4)
)

fig, axes = plt.subplots(3, 1, figsize=(10.8, 9.6), dpi=DPI, sharex=True)
x = np.arange(len(if_tc))
w = 0.24
for ax, keys, ylab, title in (
    (axes[0], ("NK_AS", "NK_IF", "NK_TC"), "L2 NK fraction", "NK  (near-zero = class not predicted)"),
    (axes[1], ("T_AS", "T_IF", "T_TC"), "L2 T fraction", "T (CD4+CD8+Treg)  AS vs IF vs TC"),
    (axes[2], ("B_plasma_AS", "B_plasma_IF", "B_plasma_TC"), "L2 B/Plasma fraction", "B + Plasma  AS vs IF vs TC"),
):
    ax.bar(x - w, if_tc[keys[0]], w, color=COMP_COLOR["AS"], label="AS")
    ax.bar(x, if_tc[keys[1]], w, color=COMP_COLOR["IF"], label="IF")
    ax.bar(x + w, if_tc[keys[2]], w, color=COMP_COLOR["TC"], label="TC")
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(if_tc["sample"], rotation=55, ha="right", fontsize=8)
fig.tight_layout()
if SAVE_FIGURES:
    save_figure(fig, fig_dir / "HCC_nature_NK_IF_vs_TC.png")
display(fig)
plt.close(fig)


### How to read the 17-slide figure (`HCC_nature_NK_IF_vs_TC.png`)

Same rim rule on every whole slide (not just the L-8012 zoom). Numbers: `figures/HCC_nature_IF_TC_L2_summary.csv`. There are still **no REC / non-REC labels**, so this is a spatial contrast only.

**Row 1 — NK.** Only `L-8002` (9,574 NK), `L-8004` (4,033) and `L-8005` (2,329) have a real NK call. The other 14 slides have ≤30 NK (L-8012 has 3). A flat bar means “L2 did not predict NK”, not “IF and TC have the same NK biology”. On the three NK-rich slides the fraction is **not** IF-specific: `L-8002` AS 7.2% > TC 5.4% > IF 4.9%; `L-8005` is even higher in TC than IF. That is the opposite of the paper’s IF-enriched NK, and it is an H&E class call, not CD56 IHC.

**Row 2 — T (CD4 + CD8 + Treg).** Several slides show T in **AS ≥ IF > TC**: `L-8003` (AS 63%, IF 37%, TC 35%), `L-8013`, `L-8015`, `L-8017`. `L-8012` is almost empty of T on the whole-slide scale (AS 2.5%, IF 1.1%, TC 0.07%), matching the zoom. T is the class Hist2Pheno can actually use as an IF/AS immune readout on this cohort.

**Row 3 — B + Plasma.** Same AS-first pattern on most OME slides (`L-8003`, `L-8010`, `L-8015`, `L-8017`). **`L-8001` is an outlier**: 88% of nuclei were labelled TC (tumour domain ate the slide) and B/Plasma in that “TC” is 37% — treat that slide’s compartments as failed, not as a plasma-cell-rich core.

**How to write it.** Across slides that have a stable AS/IF/TC split, predicted T and B are enriched toward the stromal side (AS, then IF), and depleted in TC. That is compatible with an immune-low tumour core on H&E, and it is **not** a replication of TIMES (five genes + NK IHC + recurrence). Quote L-8002/8004/8005 if you mention NK; do not average NK over the 14 empty slides.
